# Role Dimension Loader

This notebook maintains the `warehouse.dim_role` dimension table.

**Purpose**: Track canonical job roles from semantic mapping

**Key Features**:
* Stable surrogate keys for roles
* Role hierarchies and career levels
* Role metadata and classifications

**Source**: `semantic.sem_job_role_map`

**Target**: `workspace.warehouse.dim_role`

In [0]:
%sql
-- Extract role data from semantic job role map
CREATE OR REPLACE TEMP VIEW role_extract AS
SELECT 
  canonical_role_id,
  canonical_role_name,
  role_category,
  role_level,
  role_family,
  seniority_level,
  is_manager,
  is_executive,
  active_flag as is_active
FROM workspace.semantic.sem_job_role_map
WHERE canonical_role_id IS NOT NULL;

In [0]:
%sql
-- Generate or maintain stable surrogate keys
CREATE OR REPLACE TEMP VIEW role_with_keys AS
SELECT
  COALESCE(d.role_sk, ROW_NUMBER() OVER (ORDER BY r.canonical_role_name) + COALESCE(max_sk, 0)) as role_sk,
  r.canonical_role_id,
  r.canonical_role_name as role_name,
  r.role_category,
  r.role_level,
  r.role_family,
  r.seniority_level,
  COALESCE(r.is_manager, FALSE) as is_manager,
  COALESCE(r.is_executive, FALSE) as is_executive,
  COALESCE(r.is_active, TRUE) as is_active,
  CURRENT_TIMESTAMP() as updated_at
FROM role_extract r
LEFT JOIN workspace.warehouse.dim_role d
  ON r.canonical_role_id = d.canonical_role_id
CROSS JOIN (
  SELECT COALESCE(MAX(role_sk), 0) as max_sk 
  FROM workspace.warehouse.dim_role
) max_keys;

In [0]:
%sql
-- Merge into target dimension (SCD Type 1)
MERGE INTO workspace.warehouse.dim_role target
USING role_with_keys source
ON target.canonical_role_id = source.canonical_role_id
WHEN MATCHED THEN UPDATE SET
  target.role_name = source.role_name,
  target.role_category = source.role_category,
  target.role_level = source.role_level,
  target.role_family = source.role_family,
  target.seniority_level = source.seniority_level,
  target.is_manager = source.is_manager,
  target.is_executive = source.is_executive,
  target.is_active = source.is_active,
  target.updated_at = source.updated_at
WHEN NOT MATCHED THEN INSERT (
  role_sk,
  canonical_role_id,
  role_name,
  role_category,
  role_level,
  role_family,
  seniority_level,
  is_manager,
  is_executive,
  is_active,
  updated_at
) VALUES (
  source.role_sk,
  source.canonical_role_id,
  source.role_name,
  source.role_category,
  source.role_level,
  source.role_family,
  source.seniority_level,
  source.is_manager,
  source.is_executive,
  source.is_active,
  source.updated_at
);

In [0]:
%sql
-- Validate role dimension
SELECT 
  COUNT(*) as total_roles,
  COUNT(DISTINCT role_category) as role_categories,
  COUNT(DISTINCT role_family) as role_families,
  SUM(CASE WHEN is_manager THEN 1 ELSE 0 END) as manager_roles,
  SUM(CASE WHEN is_executive THEN 1 ELSE 0 END) as executive_roles,
  SUM(CASE WHEN is_active THEN 1 ELSE 0 END) as active_roles
FROM workspace.warehouse.dim_role;

-- Sample roles
SELECT 
  role_sk,
  role_name,
  role_category,
  seniority_level,
  is_manager,
  is_active
FROM workspace.warehouse.dim_role
ORDER BY role_category, seniority_level, role_name
LIMIT 20;